In [22]:
import pandas as pd
import numpy as np

import altair as alt

# Using ecostyles package (installed in environment)
from ecostyles import EcoStyles
styles = EcoStyles()
styles.register_and_enable_theme(theme_name='cotd')

In [68]:
aging = pd.read_csv('unpopulation_dataportal_20260729110938.csv')
aging.columns.tolist()

columns = [
 'IndicatorShortName',
 'LocationId',
 'Location',
 'Time',
 'EstimateMethod',
 'Value']

aging = aging[columns]

aging["Location"] = aging["Location"].replace(
    "United States of America", "USA"
)
aging['Time'] = aging["Time"].astype(str)
aging["Time"] = pd.to_datetime(aging["Time"], errors="coerce")

#label where projection method is used
proj_start = 2024
max_year = aging["Time"].max()


In [69]:
# --- explicit color mapping to avoid similar colors ---
countries = sorted(aging["Location"].unique())
color_scale = alt.Scale(
    domain=countries,
    range=[ "#179fdb", "#e6224b", "#f4c245","#eb5c2e", "#36b7b4","#0063AF"]
)

x_min = aging["Time"].min()
x_max = aging["Time"].max()
x_scale = alt.Scale(domain=[x_min, x_max])

print(x_min, x_max)

1990-01-01 00:00:00 2026-01-01 00:00:00


In [ ]:
aging = aging[aging["Time"] >= pd.Timestamp("1990-01-01")].copy()

proj_start = (
    aging[aging["EstimateMethod"] == "Projection"]
    .groupby("Location")["Time"].min()
    .min()
)

x_min1 = pd.Timestamp("1990-01-01")   # hardcoded start
x_max1 = aging["Time"].max()
x_scale = alt.Scale(domain=[x_min1, x_max1])

shade = alt.Chart(pd.DataFrame({
    "start": [proj_start],
    "end": [x_max1]
})).mark_rect(opacity=0.08, color="gray").encode(
    x=alt.X("start:T", scale=x_scale),
    x2="end:T"
)

rule = alt.Chart(pd.DataFrame({"x": [proj_start]})).mark_rule(
    color="gray", strokeDash=[4, 4]
).encode(
    x=alt.X("x:T", scale=x_scale)
)

line = alt.Chart(aging).mark_line(point=alt.OverlayMarkDef(size=40, filled=True)).encode(
    x=alt.X("Time:T", scale=x_scale),
    y=alt.Y("Value:Q", title="Old-Age Dependency Ratio (%)"),
    color=alt.Color("Location:N", scale=color_scale, legend=None),
    tooltip=["Location", "Time", "Value"]
)

last_points = aging.loc[aging.groupby("Location")["Time"].idxmax()].copy()

label_offsets = {
    "Canada": .6,
    "United Kingdom": -.5,
}
last_points["label_y"] = last_points["Value"] + last_points["Location"].map(label_offsets).fillna(0)

country_labels = alt.Chart(last_points).mark_text(
    align="left", dx=6, baseline="middle", fontSize=11
).encode(
    x=alt.X("Time:T", scale=x_scale),
    y=alt.Y("label_y:Q"),
    text="Location:N",
    color=alt.Color("Location:N", scale=color_scale, legend=None)
)

y_min1 = aging["Value"].min()
y_max1 = aging["Value"].max()

axis_labels1 = pd.DataFrame({
    "x": [x_min1, x_min1],
    "y": [y_max1, y_min1],
    "text": ["More dependent →", "← Less dependent"]
})

annotation1 = alt.Chart(axis_labels1).mark_text(
    angle=270,
    align="center",
    baseline="middle",
    dy=-30,
    dx=-20,
    fontSize=11,
    color="gray"
).encode(
    x=alt.X("x:T"),
    y=alt.Y("y:Q"),
    text="text:N"
)

chart1 = (shade + rule + line + country_labels + annotation1).properties(
    width=450,
    height=350,
    title=alt.TitleParams(
        text="Old-Age Dependency Ratio (OADR) Over Time by Country",
        subtitle=["OADR is the population aged 65+ per 100 people aged 16-64.","Shaded area indicates projected values."],
        offset=7
    )
)

chart1 = styles.add_source(chart1, ["Source: UN World Population Prospects 2024"])

chart1 = chart1.configure_view(
    strokeWidth=0, clip=False
).configure_axis(
    grid=False
)

styles.save(chart1, 'charts', 'OADR_countriescomparison', width=400, height=500)
chart1

alt.LayerChart(...)

In [26]:
#UK OADR projections


#function parsing the ONS variant sheets to get OADR by year in long format

def parse_ons_variant_sheet(filepath, sheet_name='PERSONS', variant_label="Zero net migration"):
    """
    Parses an ONS national population projections variant worksheet
    (the 'Persons, thousands' stacked-table format) and returns
    OADR by year in long format.
    """
    # Read raw, no header assumptions -- these sheets have metadata rows on top
    raw = pd.read_excel(filepath, sheet_name=sheet_name, header=None)

    # Find the row where "Table 3" starts
    table3_start = raw[raw.apply(
        lambda row: row.astype(str).str.contains("Table 3", case=False, na=False).any(),
        axis=1
    )].index[0]

    # Header row (years) sits right after the "Table 3" title row
    header_row = table3_start + 1
    years = raw.iloc[header_row, 1:].dropna().astype(int).tolist()

    # Grab everything below the header until the sheet ends or hits blank rows
    table3 = raw.iloc[header_row + 1:].reset_index(drop=True)
    table3.columns = ["Metric"] + years
    table3 = table3.dropna(subset=["Metric"])

    # Pull just the two rows we need for OADR
    working_age = table3[table3["Metric"].str.strip() == "16 to 64 [note 5]"]
    pension_age = table3[table3["Metric"].str.strip() == "65 and over [note 5]"]

    if working_age.empty or pension_age.empty:
        raise ValueError("Couldn't find '16 to 64' / '65 and over' rows -- check row label spelling in this file.")

    working_age = working_age.iloc[0, 1:].astype(float)
    pension_age = pension_age.iloc[0, 1:].astype(float)

    oadr = (pension_age / working_age) * 100  # standard 65+/16-64 convention

    result = pd.DataFrame({
        "Time": years,
        "Value": oadr.values,
        "Variant": variant_label
    })

    return result



In [38]:
principal = parse_ons_variant_sheet('ukpppsummary.xlsx', variant_label="Principal")
zeronetmigration = parse_ons_variant_sheet('ukppzsummary.xlsx', variant_label="Zero net migration")
combined = pd.concat([principal, zeronetmigration], ignore_index=True)
combined["Variant"] = combined["Variant"].replace({
    "Principal": "Current projected migration",
    "Zero net migration": "Zero net migration"
})

combined['Time'] = combined["Time"].astype(str)

In [27]:
combined

,Time,Value,Variant
0,1990,30.240265,Current projected migration
1,1991,30.760470,Current projected migration
2,1992,31.328560,Current projected migration
3,1993,31.901484,Current projected migration
4,1994,32.509314,Current projected migration
...,...,...,...
197,2002,81.908704,Zero net migration
198,2003,82.240867,Zero net migration
199,2004,82.551469,Zero net migration
200,2005,82.839344,Zero net migration


In [77]:
y_min = combined["Value"].min() - 7
y_max = combined["Value"].max() + 6

x_min = combined["Time"].min()
x_max = combined["Time"].max()
x_scale = alt.Scale(domain=[x_min, x_max])

axis_labels = pd.DataFrame({
    "x": [x_min, x_min],
    "y": [y_max, y_min],
    "text": ["More dependent →", "← Less dependent"]
})

annotation = alt.Chart(axis_labels).mark_text(
    angle=270,
    align="center",
    baseline="middle",
    dy=-30,
    dx=-60,
    fontSize=11,
    color="gray"
).encode(
    x=alt.X("x:T", scale=x_scale),
    y=alt.Y("y:Q"),
    text="text:N"
)

# --- main line, no legend ---
line = alt.Chart(combined).mark_line(strokeWidth=3).encode(
    x=alt.X("Time:T", title=None, scale=x_scale),
    y=alt.Y("Value:Q", title="Old-Age Dependency Ratio (%)"),
    color=alt.Color("Variant:N", legend=None),
    tooltip=["Time", "Value", "Variant"]
)

# --- direct label at the end of each line ---
last_points = combined.loc[combined.groupby("Variant")["Time"].idxmax()]

line_labels = alt.Chart(last_points).mark_text(
    align="left", dx=6, baseline="middle", fontSize=12, fontWeight="bold"
).encode(
    x=alt.X("Time:T", scale=x_scale),
    y="Value:Q",
    text="Variant:N",
    color=alt.Color("Variant:N", legend=None)
)

chart2 = (annotation + line + line_labels).properties(
    width=400,
    height=500,
    title=alt.TitleParams(
        text=["UK Old-Age Dependency (OADR) Ratio Projections:", "current projected migration vs. zero net migration"],
        subtitle="OADR is the population aged 65+ per 100 people aged 16-64. ",
        offset=7
    )
)
    

chart2 = styles.add_source(chart2, ["Source: Projections calculated by ONS and published in April 2026."])


chart2
styles.save(chart2, 'charts', 'OADR_ukprojections', width=400, height=500)

In [53]:
chart2

alt.LayerChart(...)

In [123]:

chart = alt.Chart(combined).mark_line(strokeWidth=3).encode(
    x=alt.X("Time:Q", axis=alt.Axis(format="d"), title=None),
    y=alt.Y("Value:Q", title="Old-Age Dependency Ratio (%)"),
    color=alt.Color(
        "Variant:N",
        title="Scenario",
        legend=alt.Legend(orient="bottom")
    ),
    tooltip=["Time", "Value", "Variant"]
).properties(
    width=700,
    height=400,
    title=alt.TitleParams(
        text="UK Old-Age Dependency (OADR) Ratio Projections: With current projected migration vs. zero net migration",
        subtitle="OADR is the population aged 65+ per 100 people aged 16-64. Projections calculated by ONS and published in April 2026.",
        offset=7
    )
)

chart

alt.Chart(...)